In [ ]:
import MDAnalysis as mda 
from pathlib import Path 

parent = Path("examples/mace_water")
data_file = parent / "system.data"
traj = parent / "lammps.dump"

# MDAnalysis universe
u = mda.Universe(
    str(data_file),
    str(traj),
    topology_format="DATA",
    format="LAMMPSDUMP",
    atom_style="id type x y z",
)
len(u.trajectory)

In [ ]:
import numpy as np
from ase.units import fs 

momenta = []
forces = []

masses = u.atoms.masses  # shape: (n_atoms,)

for ts in u.trajectory[-1000:]:
    # ts.velocities and ts.forces are (n_atoms, 3)
    momenta.append(masses[:, None] * ts.velocities / (1e3 * fs))  # lammps velocites are in A/ps --> convert to ASE units
    forces.append(ts.forces)

momenta = np.array(momenta)  # shape: (n_frames, n_atoms, 3)
forces = np.array(forces)    # shape: (n_frames, n_atoms, 3)

np.save(parent / "momenta.npy", momenta)
np.save(parent / "forces.npy", forces)


In [ ]:
fs

In [ ]:
import numpy as np 

forces = np.load(parent / "forces.npy")
momenta = np.load(parent / "momenta.npy")

forces.shape, momenta.shape

In [ ]:
forces_reshaped = forces.reshape(forces.shape[0], -1)
momenta_reshaped = momenta.reshape(momenta.shape[0], -1)

n_frames = forces_reshaped.shape[0]
n_features = forces_reshaped.shape[1]

forces_outer_mean = np.zeros((n_features, n_features), dtype=forces_reshaped.dtype)
momenta_outer_mean = np.zeros((n_features, n_features), dtype=momenta_reshaped.dtype)

for i in range(n_frames):
    forces_outer_mean += np.outer(forces_reshaped[i], forces_reshaped[i])
    momenta_outer_mean += np.outer(momenta_reshaped[i], momenta_reshaped[i])

forces_outer_mean /= n_frames
momenta_outer_mean /= n_frames

forces_outer_mean.shape, momenta_outer_mean.shape

In [ ]:
import matplotlib.pyplot as plt 
%matplotlib widget 

plt.figure(figsize=(5, 5))
plt.imshow(forces_outer_mean, aspect='auto', cmap='inferno')
plt.colorbar(label='Value')
plt.axis('equal')
plt.show()

In [ ]:
np.max(forces_outer_mean)

In [ ]:
from scipy.linalg import eig 

eigenvalues, eigenvectors = eig(a=forces_outer_mean, b=momenta_outer_mean)

In [ ]:
from scipy.constants import speed_of_light

wavenumbers = np.sqrt(eigenvalues.real) * fs * 1e15 / speed_of_light / 100
wavenumbers

In [ ]:
import matplotlib.pyplot as plt 
%matplotlib widget 

fig, ax = plt.subplots()
ax.hist(wavenumbers, bins=np.linspace(0, 1e4, 1000))
ax.set_xlim([0, 2e3])
# ax.set_xlim([-1e9, 1e9])

In [ ]:
np.where((wavenumbers > 1500) * (wavenumbers < 2000))[0][0]

In [ ]:
masses_repeated = np.repeat(masses, 3)
Ztransinv = np.diag(masses_repeated) @ eigenvectors
Zinv = Ztransinv.T
q = eigenvectors[:, 5]
x = np.linalg.inv(Zinv) @ q

x

In [ ]:
from ase.io import read 
from ase.visualize import view 
atoms = read(parent / 'lammps.dump', index=-500)
omega = np.sqrt(eigenvalues.real[5])
traj = []

for t in range(1000):
    a = atoms.copy()
    a.positions = atoms.positions + 0.5 * x.reshape(-1, 3) * np.sin(omega * t)
    traj.append(a)

view(traj)

192 intramolecular vibrational modes
6 rotational and translational modes for center of mass
576 